# qwanarings example

The `qwanarings` command-line utility is used to automatically detect ring boundaries in wood anatomy images whose cells have already been measured using `qwanaflow`.

## Running from the command-line

Typically, `qwanarings` will be run on the output directory of `qwanaflow` as follows:

In [ ]:
# Here, the exclamation mark indicates that this command is run by the shell and not by python
# Indeed, qwanarings should be launched from the command line
!qwanarings --input_dir test_output

The `qwanarings` command supports the following arguments which can be viewed when providing the `--help` argument:

In [ ]:
!qwanarings --help

## Detailed qwanarings usage

The remainder of this document explains in detail how `qwanarings` works behind the scences to detect ring boundaries. This may be useful to users who want better control over the workflow or would like to leverage these functions individually in their analyses.

### Setting up the required libraries and variables

The following Python libraries need to be imported for the following code to work. They are automatically imported when you run `qwanarings` from the command-line.

In [ ]:
from qwanamiz import rings_functions as qrings # main qwanarings functions
import numpy as np                             # library for arrays and matrics
import pandas as pd                            # manipulating data tables as DataFrames
from matplotlib import pyplot as plt           # plotting functions
import networkx as nx                          # graph structures

A few variables that you would usually pass as command-line arguments are also necessary to make the code work:

In [ ]:
pix_to_um = 0.55042690590734                                 # the size per pixel in micrometers
prefix = "test_output/test_image.png_outputs/test_image.png" # a prefix for the input/output files to use
firstyear = 1                                                # the year of the first ring, defautls to 1 (unknown)
mincells = 4                                                 # the minimum number of cells in a boundary region to consider it

Now, we can read in the outputs from `qwanaflow`, which are used as inputs to `qwanarings`:

In [ ]:
# Setting the paths to the input files
imgs_path = f"{prefix}_imgs.npz"           # various raster images (most importantly cell labels)
cells_path = f"{prefix}_cells.csv"         # cell measurements and related metadata
adjacency_path = f"{prefix}_adjacency.csv" # cell adjacencies (graph) and related metadata

# Reading data from the files
images = np.load(imgs_path)             # images as compressed numpy arrays
celldata = pd.read_csv(cells_path)      # pandas DataFrame of cell measurements
adjacency = pd.read_csv(adjacency_path) # pandas DataFrame of cell adjacencies

# Giving an explicit variable name to the expanded labels image (array of whole cell areas including lumen and cell wall)
expanded_labels = images['explabs']

# Doing the same thing for the array of cell labels that does not include their cell wall
lumen_labels = images['labs']

# Explicitly setting the double index on the adjacency DataFrame
adjacency.set_index(['label1', 'label2'], inplace=True)
    

As a reminder from the `qwanaflow` detailed workflow, here is the image that we will be working with :

<img src="../tests/data/test_image.png" width=500 height=500>

The labeled image shows the lumens of the cells identified by `qwanaflow`:

In [ ]:
plt.imshow(lumen_labels)

### Identifying cells involved in ring transitions

The first step in finding tree-ring boundaries is to determine where transitions from latewood to earlywood occur. Indeed, the last cells of any given year are latewood whereas the first cells of a year are earlywood. By finding out where that transition occurs, it should be possible to draw the boundaries between years.

`qwanarings` relies on the commonly used Mork's index to classify cells as either latewood or earlywood, that is, if a cell's wall thickness is at least one quarter that of its diameter, it is considered latewood, otherwise it is considered earlywood. The function `morks_index` accordingly adds a column called `'woodzone'` to the DataFrame of cell metadata to indicate its type.

In [ ]:
# Classifying cells as early- or latewood based on Mork's index
celldata = qrings.morks_index(celldata)

# The classification is added to column 'woodzone'
celldata['woodzone'].value_counts()

Visualizing which cells were identified as either type confirms the relevance of Mork's index for this task:

In [ ]:
# Extracting the labels of earlywood and latewood cells
earlywood_labels = celldata[celldata['woodzone'] == 'earlywood']['label'].values
latewood_labels  = celldata[celldata['woodzone'] == 'latewood']['label'].values

plt.imshow(np.isin(lumen_labels, earlywood_labels) + 2 * (np.isin(lumen_labels, latewood_labels)))

Using that classification, we can leverage the region adjacency graph (a data structure that tells use which cells are adjacent to each other) to identify cells that we define as "last cells" and "right cells". "Last cells" are the cells that were last formed in a given radial file for a given year, and are therefore latewood cells. "Right cells" are the cells directly adjacent to the right of the last cells, and are the first (earlywood) cells formed in a given year.

This classification as "last cells" and "right cells" is achieved with the `get_lastcells` function. In addition to identifying last cells as latewood cells directly adjacent to an earlywood cell, the following two conditions have to be met for a cell to be considered a "last cell" :

* the diameter of the last cell, multipled by some factor (`diameter_factor`, defaults to 2.5) must be smaller than that of the cell to its right
* similarly, the diameter of the last cell (multiplied by `diameter_factor_prev`, defaults to 8) must be greater than that of the cell to its left

In [ ]:
# Get lastcells in rings based on diameter and woodzone cell features
lastcells_labels, rightcells_labels, _ = qrings.get_lastcells(celldata, adjacency)

# Create a mask where pixels belong to lastcells or their right_neighbors
rightcells_mask = np.zeros_like(expanded_labels, dtype=bool)
rightcells_mask[np.isin(expanded_labels, rightcells_labels)] = True

# We can have a look at which cells were identified as "right cells"
plt.imshow(images['bw_img'], cmap = 'gray')
plt.imshow(rightcells_mask, alpha = 0.5)

The next step involves grouping the cells located at boundaries (last cells and right cells) together according to the boundary that they define. To do this, we leverage functionality from the `networkx` module to build a graph that will store adjacencies between cells located at boundaries.

In [ ]:
# Extracting a DataFrame of right cells
rightcells_df = celldata[celldata["label"].isin(rightcells_labels)].copy()

# Re-formatting rightcells_labels as a set
rightcells_labels = set(rightcells_df["label"])

# Crearing a graph using previously filtered nodes and edges
# A node is created for each last cell and right cell
# An edge is created for each adjacency between any of those cells (right-right, last-last, and last-right)
graph = qrings.boundary_graph(celldata, adjacency, lastcells_labels, rightcells_labels)

# A function to visualize adjacencies stored in the graph
def plot_graph(cell_df, adj_graph, pixel_size):
    cell_copy = cell_df.copy()
    cell_copy.set_index('label', inplace = True)
    for i in adj_graph.edges:
        x1 = cell_copy.loc[i[0]]['centroid-1']
        y1 = cell_copy.loc[i[0]]['centroid-0']
        x2 = cell_copy.loc[i[1]]['centroid-1']
        y2 = cell_copy.loc[i[1]]['centroid-0']

        plt.plot(np.array([x1, x2]) / pixel_size, np.array([y1, y2]) / pixel_size, c = 'blue')

    return

# Showing the adjacenencies as blue lines on top of the cell array
plt.figure(figsize = (12, 12))
plt.imshow(images['bw_img'], cmap = 'gray')
plt.imshow(rightcells_mask, alpha = 0.5)
plot_graph(celldata, graph, pix_to_um)


We can now use this graph structure identify cell groups that can be reached from each other using the graph. In graph parlance, these groups are called "connected components", and the `networkx` module conveniently provides the `connected_components` function for computing those.

In [ ]:
# Find connected components (output is a list of sets)
connected_components = list(nx.connected_components(graph))
    
# Create a dictionary mapping nodes to component IDs
node_to_component = {}
for i, component in enumerate(connected_components):
    for node in component:
        node_to_component[node] = i + 1 # i + 1 to avoid the 0 label reserved for background
    
# Prepare a mask with cells that are part of a boundary
target_labels = np.array(list(node_to_component.keys()))
target_mask = np.isin(expanded_labels, target_labels)
    
# Get region IDs for those cell labels
label_array = expanded_labels[target_mask]
    
# Creating an array that labels cells according to their boundary region
region_array = np.vectorize(node_to_component.get)(label_array)
boundary_array = np.zeros_like(expanded_labels, dtype=int)
boundary_array[target_mask] = region_array

The boundary regions are therefore identified as follows, with each boundary region identified with a different color:

In [ ]:
boundary_tmp = boundary_array.astype(float)
boundary_tmp[boundary_tmp == 0] = np.nan
plt.imshow(images['bw_img'], cmap = 'gray')
plt.imshow(boundary_tmp, alpha = 0.8)

We have successfully grouped cells according to the ring boundary that they define, but as we can see, the result is rather fragmented as some cells were not properly recognized as being located at a boundary. The next step for `qwanarings` is to join those boundary regions together such that they can be used for defining growth rings.

### Expanding and merging ring-transition boundaries

### Refining and filtering ring-transition boundaries

### Assigning cells to tree rings

### Ring width and related measurements

### Outputs

### 